In [16]:
from ..middleWare import *
load_dotenv(override=True)

DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')
model=init_chat_model(
    model='deepseek-v4-flash',
    model_provider='deepseek',
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={'thinking':{"type":'disabled'}},
)

In [17]:
#定义Todo工具列表
@tool
def search_flights(origin: str, destination: str, date: str) -> str:
    '''
    搜索航班信息

    Args:
        origin:出发地
        destination:目的地
        date:出发日期
    '''
    return f"已找到从 {origin} 到 {destination} 于 {date} 的航班，价格 3000 元"

@tool
def search_hotels(city: str) -> str:
     '''
    搜索酒店信息

    Args:
        city:城市
    '''
     return f"已找到 {city}的酒店，价格 500 元/晚"

@tool
def book_travel_insurance(name: str, destination: str) -> str:
     '''
    预定旅行保险

    Args:
        name:受保人姓名
        destination:目的地
    '''
     return f"已为 {name} 预订前往 {destination} 的旅行保险"


In [18]:
#使用TodoList中间件，结合工具拆解任务按顺序调用，先调用内置的write_todos工具拆解出子任务，然后按照子任务要求调用外部工具
#每阶段结束会返回todos阶段进度，包含拆解出的子任务内容content和对应进度status：completed已完成，in_progress进行中，pending待办
myagent=create_agent(
    model=model,
    tools=[search_flights, search_hotels, book_travel_insurance],
    middleware=[
        TodoListMiddleware(),
    ],
    system_prompt="你是一个旅行规划助手，擅长将复杂的旅行需求拆解成可执行的步骤。"
)
response=myagent.invoke({
    'messages':HumanMessage('帮我规划一次从北京到新加坡的商务旅行，时间是7月25日到7月30日，我需要预订航班、酒店和旅行保险。')
})
rprint(response)

{
    'messages': [
        HumanMessage(
            content='帮我规划一次从北京到新加坡的商务旅行，时间是7月25日到7月30日，我需要预订航班、酒店和旅行保险。
',
            additional_kwargs={},
            response_metadata={},
            id='45b4fab1-e747-4704-86ad-e876db215879'
        ),
        AIMessage(
            content='我来帮你规划这次商务旅行！让我先创建一个任务清单，然后逐步执行。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 150,
                    'prompt_tokens': 1611,
                    'total_tokens': 1761,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1536},
                    'prompt_cache_hit_tokens': 1536,
                    'prompt_cache_miss_tokens': 75
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '4f8c6e0a-c00a-4f17-bbe9-29a68949c6e9',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f91ec-5aad-74e1-bf15-c85a65085372-0',
            tool_calls=[
                {
                    'name': 'write_todos',
                    'args': {
                        'todos': [
                            {'content': '搜索7月25日北京到新加坡的航班', 'status': 'in_progress'},
                            {'content': '搜索7月30日新加坡到北京的航班', 'status': 'pending'},
                            {'content': '搜索新加坡的酒店', 'status': 'pending'},
                            {'content': '预订旅行保险', 'status': 'pending'},
                            {'content': '整理最终旅行方案', 'status': 'pending'}
                        ]
                    },
                    'id': 'call_00_7V4PrVvoakNkNu7cbOEf0825',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1611,
                'output_tokens': 150,
                'total_tokens': 1761,
                'input_token_details': {'cache_read': 1536},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Updated todo list to [{'content': '搜索7月25日北京到新加坡的航班', 'status': 'in_progress'}, 
{'content': '搜索7月30日新加坡到北京的航班', 'status': 'pending'}, {'content': '搜索新加坡的酒店', 'status': 
'pending'}, {'content': '预订旅行保险', 'status': 'pending'}, {'content': '整理最终旅行方案', 'status': 
'pending'}]",
            name='write_todos',
            id='21cad2ad-755e-4938-b7be-4d81836624c3',
            tool_call_id='call_00_7V4PrVvoakNkNu7cbOEf0825'
        ),
        AIMessage(
            content='好的，现在先搜索7月25日北京到新加坡的航班，以及新加坡的酒店信息。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 133,
                    'prompt_tokens': 1854,
                    'total_tokens': 1987,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1536},
                    'prompt_cache_hit_tokens': 1536,
                    'prompt_cache_miss_tokens': 318
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '64f40d6c-b980-4241-a7bf-9ae41d9369b0',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f91ec-5fdc-7881-b41d-278620a9cb81-0',
            tool_calls=[
                {
                    'name': 'search_flights',
                    'args': {'origin': '北京', 'destination': '新加坡', 'date': '2025-07-25'},
                    'id': 'call_00_h1jlofdL1AoGqOQwKcno8910',
              